# Hardware-test preflight setup

Run this notebook **before** `pytest tests/hardware --scope <scope>` to:

1. Instantiate the selected microscope with its real Micro-Manager cfg.
2. Open napari-micromanager sharing the same MMCore so you can live-view,
   focus, and try channels interactively.
3. Pick imaging / stim / optocheck channels from the cfg's actual groups.
4. Verify DMD-to-camera focus with a checkerboard snap.
5. Measure camera baseline (DMD off) and bright (DMD all-on) so the
   pytest tests can auto-calibrate their brightness thresholds instead
   of hardcoding them.

Results land in `tests/hardware/.preflight.json`, which `conftest.py`
reads at session start. Re-run this notebook whenever you change the
sample, LED power, binning, or camera ROI.

In [ ]:
# --- Cell 1: pick the scope ---
SCOPE = "moench"  # moench | niesen | jungfrau
PREFLIGHT_PATH = "tests/hardware/.preflight.json"

import json
import weakref
from pathlib import Path

import numpy as np
import pymmcore_plus.core._mmcore_plus as _mmcore_mod

In [ ]:
# --- Cell 2: instantiate scope, share its mmc with napari-mm's singleton ---
if SCOPE == "moench":
    from faro.microscope.pertzlab.moench import Moench
    mic = Moench(affine_calibration_matrix=np.eye(3))
elif SCOPE == "niesen":
    from faro.microscope.pertzlab.niesen import Niesen
    mic = Niesen(affine_calibration_matrix=np.eye(3), fast_init=True)
elif SCOPE == "jungfrau":
    from faro.core.dmd import DMD
    from faro.microscope.pertzlab.jungfrau import Jungfrau
    mic = Jungfrau()
    # Jungfrau cfg may not declare an SLM device yet
else:
    raise ValueError(f"unknown SCOPE {SCOPE!r}")

# napari-micromanager calls CMMCorePlus.instance() internally and gets a
# fresh core if none is registered. Register ours so the viewer shares
# state with the notebook.
_mmcore_mod._instance = weakref.ref(mic.mmc)
print(f"{SCOPE}: {mic.mmc.getLoadedDevices()}")

In [ ]:
# --- Cell 3: launch napari with the micromanager widget ---
# The MainWindow will pick up mic.mmc via the singleton we registered.
# Use napari's live-view / snap / channel presets to focus and preview.
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
win = MainWindow(viewer)
viewer.window.add_dock_widget(win, name="Micro-Manager")
print("Use napari-mm to focus and preview. Come back to the notebook")
print("once the sample is in focus and you know which channels to use.")

## Pick channels

After focusing in napari-mm, fill in the channel names below. The lists
come straight from the cfg so you know what's available. Exposures /
LED power can still be tuned later â€” they aren't written to
`.preflight.json` until the final cell.

In [ ]:
# --- Cell 4: enumerate available config groups + presets ---
for g in mic.mmc.getAvailableConfigGroups():
    presets = list(mic.mmc.getAvailableConfigs(g))
    print(f"  {g}: {presets}")

In [ ]:
# --- Cell 5: pin the selections. Edit these to match your setup. ---
CHANNEL_GROUP = "TTL_ERK"
IMAGING_CHANNEL = "mScarlet3"
OPTOCHECK_CHANNEL = "mCitrine"
STIM_CHANNEL = "CyanStim"
IMAGING_EXPOSURE_MS = 100.0
OPTOCHECK_EXPOSURE_MS = 100.0
STIM_EXPOSURE_MS = 100.0
STIM_POWER = 5  # LED level for the stim channel (e.g. Cyan_Level)

# Sanity: every named channel must exist in the cfg
_presets = set(mic.mmc.getAvailableConfigs(CHANNEL_GROUP))
for name in (IMAGING_CHANNEL, OPTOCHECK_CHANNEL, STIM_CHANNEL):
    assert name in _presets, f"{name!r} not in group {CHANNEL_GROUP!r}: {_presets}"
print("channels OK")

## Live DMD focus aid

This cell latches a checkerboard onto the DMD and primes the stim
channel. Click **Live** in napari-mm after running it — the camera
will stream frames with the checker projected, so you can refocus
until the squares are crisp. When done, stop Live in napari-mm and
run the disarm cell below to blank the DMD.

On scopes with ``Mosaic3 OverlapMode=On`` the DMD only fires during
camera TTL pulses; napari's Live view produces those pulses, so the
pattern appears as soon as Live starts. If it doesn't, flip
``OverlapMode`` to ``Off`` via the napari-mm property browser and
retry.

In [ ]:
# --- Cell 6a: arm DMD checkerboard for live focus ---
mic.mmc.setConfig(CHANNEL_GROUP, STIM_CHANNEL)
mic.mmc.setExposure(STIM_EXPOSURE_MS)
mic.mmc.setSLMExposure(mic.dmd.name, STIM_EXPOSURE_MS)
# Resolve the LED power property for this scope/channel. On
# Moench/Niesen this is ("LED", "Cyan_Level"); elsewhere use
# whatever the stim channel's power property is in the cfg.
try:
    mic.mmc.setProperty("LED", "Cyan_Level", STIM_POWER)
except Exception as e:
    print(f"(skipping LED power set: {e})")
mic.dmd.checker_board(pixels=100)
print("DMD checker armed. Press Live in napari-mm and refocus.")
print("Run the next cell to disarm when the pattern is sharp.")

In [ ]:
# --- Cell 6b: disarm DMD ---
mic.dmd.all_off()
print("DMD blanked.")

## Confirmation snap

One-shot via the same MDA path the pytest tests use — confirms the
checker is still crisp after disarming + re-arming through the full
event machinery.

In [ ]:
# --- Cell 6: checker-board focus aid ---
from threading import Event
from useq import MDAEvent, PropertyTuple
from faro.core._useq_compat import SLMImage
import matplotlib.pyplot as plt


def snap_via_mda(mask, channel, exposure_ms, power=None, power_device=None, power_prop=None):
    """Snap one camera frame while the DMD projects ``mask``.

    Runs through the MDA event machinery so the LED / DMD / camera TTL
    handshake matches what the pytest tests use.
    """
    props = []
    if power is not None:
        props.append(PropertyTuple(
            device_name=power_device, property_name=power_prop, property_value=power
        ))
    event = MDAEvent(
        index={"t": 0, "p": 0},
        channel={"config": channel, "group": CHANNEL_GROUP},
        exposure=exposure_ms,
        min_start_time=0,
        properties=props or None,
        slm_image=SLMImage(device=mic.dmd.name, exposure=exposure_ms, data=mask),
        metadata={"stim": True},
    )
    done, captured = Event(), {}
    def on_frame(img, _evt, *_):
        captured["img"] = np.asarray(img)
        done.set()
    mic.connect_frame(on_frame)
    thread = mic.run_mda(iter([event]))
    done.wait(timeout=15)
    thread.join(timeout=10)
    mic.disconnect_frame(on_frame)
    return captured["img"]


h, w = mic.dmd.height, mic.dmd.width
checker = ((np.indices((h, w)) // 100).sum(axis=0) % 2).astype(np.uint8) * 255
img = snap_via_mda(
    checker, STIM_CHANNEL, STIM_EXPOSURE_MS,
    power=STIM_POWER, power_device="LED", power_prop="Cyan_Level",
)
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap="gray")
plt.title(f"checker snap  mean={img.mean():.0f}  max={img.max()}")
plt.axis("off")

## Brightness calibration

Snap one all-off and one all-on DMD frame, measure p99, and derive the
thresholds the brightness-based stim tests use.

In [ ]:
# --- Cell 7: baseline vs bright ---
blank = np.zeros((h, w), dtype=np.uint8)
allon = np.full((h, w), 255, dtype=np.uint8)

dark_img = snap_via_mda(blank, STIM_CHANNEL, STIM_EXPOSURE_MS,
                        power=STIM_POWER, power_device="LED", power_prop="Cyan_Level")
bright_img = snap_via_mda(allon, STIM_CHANNEL, STIM_EXPOSURE_MS,
                          power=STIM_POWER, power_device="LED", power_prop="Cyan_Level")

baseline_p99 = float(np.percentile(dark_img, 99))
bright_p99 = float(np.percentile(bright_img, 99))
# Midpoint-ish threshold for "DMD fired" â€” generous enough to survive
# sample/LED drift within a session.
bright_min_p99 = (baseline_p99 + bright_p99) / 2
# Ratio for the "frame 0 in previous mode is dark" contract; guard
# against a zero baseline.
bright_vs_dark_ratio = bright_p99 / max(baseline_p99, 1.0) / 2

print(f"baseline p99 (DMD off): {baseline_p99:.0f}")
print(f"bright   p99 (DMD on):  {bright_p99:.0f}")
print(f"BRIGHT_MIN_P99       -> {bright_min_p99:.0f}")
print(f"BRIGHT_VS_DARK_RATIO -> {bright_vs_dark_ratio:.1f}")
assert bright_p99 > 3 * baseline_p99, (
    "DMD all-on barely brighter than DMD off â€” check focus, LED power, "
    "stim channel, or whether room lights are on."
)

In [ ]:
# --- Cell 8: persist preflight JSON ---
x, y = mic.mmc.getXYPosition()
try:
    z = mic.mmc.getPosition()
except Exception:
    z = None

preflight = {
    "scope": SCOPE,
    "channel_group": CHANNEL_GROUP,
    "imaging_channel": IMAGING_CHANNEL,
    "imaging_exposure": IMAGING_EXPOSURE_MS,
    "optocheck_channel": OPTOCHECK_CHANNEL,
    "optocheck_exposure": OPTOCHECK_EXPOSURE_MS,
    "stim_channel": STIM_CHANNEL,
    "stim_exposure": STIM_EXPOSURE_MS,
    "stim_power": STIM_POWER,
    "baseline_p99": baseline_p99,
    "bright_p99": bright_p99,
    "bright_min_p99": bright_min_p99,
    "bright_vs_dark_ratio": bright_vs_dark_ratio,
    "origin": {"x": x, "y": y, "z": z},
}
Path(PREFLIGHT_PATH).write_text(json.dumps(preflight, indent=2))
print(f"wrote {PREFLIGHT_PATH}")
print(json.dumps(preflight, indent=2))

## Release hardware

Call ``mic.shutdown()`` before closing the notebook. Without it the
Python process hangs at exit waiting on device handles (same class
of bug as ``docs/napari_mm_device_leak.md``). Running this cell is
mandatory — a stuck kernel holding the Mosaic3 will block the
subsequent ``pytest`` run until you reboot.

In [ ]:
# --- Cell 9: release hardware ---
mic.shutdown()
print("mic shut down — safe to close the kernel and run pytest now.")

Now run `pytest tests/hardware --scope <scope>` from a fresh terminal.
**Close napari-mm first** â€” see `docs/napari_mm_device_leak.md` for why.